# Étude Approfondie : Infrastructures de Recharge (IRVE)

## 1. Introduction et Objectifs

Ce notebook est dédié à l'analyse des données IRVE (Infrastructures de Recharge pour Véhicules Électriques). Contrairement aux données INSEE, ces données sont "sales" et nécessitent un traitement important avant de pouvoir être agrégées à l'échelle communale.

L'objectif final de ce notebook est de construire un jeu de données agrégé par code géographique à partir de la base nationale des bornes de recharge électrique (IRVE), afin d'expliquer ou prédire le taux de véhicules électriques local.

## 2. Initialisation des données

Ici, chaque ligne correspond à une borne / point de recharge déclaré. L'objectif final étant une modélisation territoriale, chaque borne doit être rattachée à une zone géographique.

Pour mener cette étude, nous devons d'abord garantir l'intégrité géographique de la base IRVE. Cette étape technique permet de compléter les codes INSEE manquants via un géocodage spatial (coordonnées GPS) afin d'assurer une base de données exhaustive pour l'analyse.

In [ ]:
import pandas as pd
from src.utils import (
    creer_gdf_irve,
    joindre_communes,
    ajouter_codes_geo
)
from src.loading import load_irve_data, charger_communes
from src.cleaning import (
    nettoyer_code_insee
)

# 1. Chargement
url_irve ="https://www.data.gouv.fr/api/1/datasets/r/eb76d20a-8501-400e-b336-d85724de5435"
df_irve = load_irve_data(url_irve)

# 2. Préparation technique (Codes Géo)
df_irve["code_insee_commune"] = df_irve["code_insee_commune"].apply(nettoyer_code_insee)
gdf_irve = creer_gdf_irve(df_irve, "consolidated_longitude", "consolidated_latitude")
gdf_communes = charger_communes()
gdf_result = joindre_communes(gdf_irve, gdf_communes)
df_irve = ajouter_codes_geo(df_irve, gdf_result)

print(f"Base chargée et géocodée : {len(df_irve)} points de charge prêts pour l'analyse.")
df_irve.sample(5)

## 3. Sélection initiale des variables

Les variables ont été retenues selon leur potentiel explicatif :
- attractivité du réseau
- accessibilité
- performance technique
- structure concurrentielle

In [ ]:
list(df_irve.columns)

Après un premier tri des variables disponibles, nous retenons celles présentant le plus d’intérêt pour la suite de l’analyse :

In [ ]:
var_interet = [
    "code_geo_total",
    "nom_operateur",
    "implantation_station",
    "nbre_pdc",
    "puissance_nominale",
    "prise_type_ef",
    "prise_type_2",
    "prise_type_combo_ccs",
    "prise_type_chademo",
    "prise_type_autre",
    "cable_t2_attache",
    "gratuit",
    "paiement_acte",
    "paiement_cb",
    "paiement_autre",
    "tarification",
    "condition_acces",
    "reservation",
    "horaires",
    "created_at"
]

df_filtre = df_irve[var_interet].copy()

## 4. Vérification des types

Les types de données fournis par la source ne sont pas respectés dans les données brutes. Plusieurs variables booléennes et temporelles sont encodées en object, nécessitant une étape de conversion avant analyse.

In [ ]:
df_filtre.info()

Nous allons désormais corriger les types de variables afin de les aligner sur ceux indiqués dans la documentation officielle de data.gouv.fr.

#### Booléens

In [ ]:
cols_bool = [
    'prise_type_ef',
    'prise_type_2',
    'prise_type_combo_ccs',
    'prise_type_chademo',
    'prise_type_autre',
    'cable_t2_attache',
    'gratuit',
    'paiement_acte',
    'paiement_cb',
    'paiement_autre',
    'reservation'
]

# Récupérer toutes les valeurs uniques
valeurs_uniques = set()
for col in cols_bool:
    valeurs_uniques.update(df_filtre[col].unique())
print(valeurs_uniques, "\n")

mapping = {
    'true': True,
    'false': False,
    '1': True,
    '0': False
}

for col in cols_bool:
    df_filtre[col] = (
        df_filtre[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(mapping)
        .astype("boolean")
    )

for col in cols_bool:
    print(f"{col} :", df_filtre[col].unique())

Après avoir identifié des incohérences de formatage dans les variables booléennes, un mapping est appliqué afin d’harmoniser leurs valeurs. Les différentes écritures observées sont ainsi converties vers un format booléen standardisé ne prenant désormais que trois modalités possibles : `True`, `False` ou `NaN`.

#### String / catégories

In [ ]:
cols_str = [
    'nom_operateur',
    'implantation_station',
    'tarification',
    'condition_acces',
    'horaires'
]

for col in cols_str:
    df_filtre[col] = (
        df_filtre[col]
        .astype("string")
    )

df_filtre.describe(include='string[python]')

Les variables `condition_acces` et `implantation_station` présentent un faible nombre de modalités distinctes. Il peut donc être pertinent de les convertir au type `category` afin d’optimiser leur stockage et leur traitement.

Avant cette transformation, nous effectuons une analyse rapide de leur distribution afin de confirmer cette hypothèse.

In [ ]:
df_irve['condition_acces'].unique()

In [ ]:
df_irve['implantation_station'].unique()

Les variables `condition_acces` et `implantation_station` présentent plusieurs anomalies de formatage, principalement liées à des problèmes d’encodage des caractères spéciaux.

Avant toute conversion en type catégoriel, une étape de nettoyage est donc réalisée afin de :

- corriger les libellés mal encodés ;
- harmoniser les écritures multiples d’une même modalité ;
- simplifier certaines catégories pour faciliter l’analyse statistique.

---

**Nettoyage de `condition_acces`**

Plusieurs variantes incorrectes du libellé **Accès libre** ont été identifiées. Elles sont regroupées sous une seule modalité standardisée :

- `Accčs libre`
- `Accs libre`
- `Acc¸s libre`
- `AccĂ¨s libre`

sont remplacées par :

- `Accès libre`

---

**Nettoyage et regroupement de `implantation_station`**

Des erreurs d’encodage ont également été corrigées dans les libellés d’implantation des bornes.

Ensuite, les modalités ont été regroupées en quatre grandes catégories interprétables.

**Récapitulatif des catégories d’implantation des bornes IRVE**

Ces catégories décrivent le lieu d’installation ou le type d’usage des bornes de recharge.

| Catégorie | Définition | Exemples | Interprétation |
|---|---|---|---|
| **Privé** | Borne située sur un espace privé, souvent réservé à certains usagers | entreprise, hôtel, supermarché, copropriété | présence d’activités économiques ou services privés |
| **Public** | Borne installée sur un parking accessible à tous | parking municipal, gare, centre-ville | volonté locale de développer la recharge publique |
| **Voirie** | Borne installée directement sur la voie publique | rue, trottoir, stationnement urbain | utile dans les zones denses sans stationnement privé |
| **Rapide** | Station dédiée à la recharge haute puissance | hub de recharge, autoroute, station rapide | usage de transit / longs trajets |

In [ ]:
replacements_acces = {
    "Accčs libre": "Accès libre",
    "Accs libre": "Accès libre",
    "Acc¸s libre": "Accès libre",
    "AccĂ¨s libre": "Accès libre"
}
df_filtre["condition_acces"] = (
    df_filtre["condition_acces"]
    .replace(replacements_acces)
    .str.strip()
)

replacements_implantation  = {
    "Parking priv rserv  la clientle": "Parking privé réservé à la clientèle",
    "Parking priv  usage public": "Parking privé à usage public"
}
df_filtre["implantation_station"] = (
    df_filtre["implantation_station"]
    .replace(replacements_implantation )
    .str.strip()
)

mapping = {
    "Parking privé réservé à la clientèle": "prive",
    "Parking privé à usage public": "prive",
    "Parking public": "public",
    "Voirie": "voirie",
    "Station dédiée à la recharge rapide": "rapide"
}
df_filtre["implantation_station_clean"] = df_filtre["implantation_station"].replace(mapping).str.strip()

In [ ]:
df_filtre['implantation_station_clean'] = df_filtre['implantation_station_clean'].astype('category')
df_filtre['condition_acces'] = df_filtre['condition_acces'].astype('category')

#### Descriptif rapide de nos variables

In [ ]:
df_filtre.describe(include='all')

## 5. Analyse des valeurs manquantes

Pour la suite de l’analyse, nous commençons par supprimer la variable `implantation_station`, devenue inutile depuis la création de sa version nettoyée et regroupée.

In [ ]:
df_filtre = df_filtre.drop(columns=["implantation_station"])

In [ ]:
na = (
    df_filtre.isna()
    .sum()
    .sort_values(ascending=False)
)

na_pct = (na / len(df_filtre) * 100).round(2)

pd.DataFrame({
    "nb_manquants": na,
    "%": na_pct
}).head(10)

Les variables comportants beaucoup de valeurs manquantes pourraient être problématiques. Analysons les :

#### tarification

Au-delà du nombre important de valeurs manquantes, cette variable contient des informations sous forme de texte libre. La grande diversité des modalités rend son exploitation difficile dans un cadre d’analyse ou de modélisation.

Par conséquent, nous faisons le choix de l’écarter de l’étude.

In [ ]:
print(df_filtre["tarification"].value_counts(dropna=False).head(15))
print()
print("Nb modalités uniques :", df_filtre["tarification"].nunique())
df_filtre = df_filtre.drop(columns=["tarification"])

#### cable_t2_attache

Cette variable présente non seulement une proportion importante de valeurs manquantes (près de la moitié des observations), mais également un fort déséquilibre dans ses modalités.

Dans ce contexte, et afin d’éviter d’introduire du bruit dans l’analyse, nous faisons le choix de l’écarter de l’étude.

In [ ]:
df_filtre["cable_t2_attache"].value_counts(normalize=True) * 100
df_filtre = df_filtre.drop(columns=["cable_t2_attache"])

#### paiement_autre
Malgré environ 15 % de valeurs manquantes, nous faisons le choix de conserver cette variable à ce stade de l’analyse, car elle nous semble potentiellement informative. Si elle se révèle finalement peu utile pour l’analyse, nous pourrons toujours la supprimer par la suite.

In [ ]:
df_filtre["paiement_autre"].value_counts(normalize=True) * 100

#### gratuit
Au-delà du nombre de valeurs manquantes, cette variable présente un fort déséquilibre entre ses modalités : la quasi-totalité des bornes étant payantes, l’information relative à la gratuité apporte peu de variance et donc peu de valeur explicative.

Dans ce contexte, et afin d’éviter d’introduire du bruit dans l’analyse, nous faisons le choix de l’écarter de l’étude.

In [ ]:
df_filtre["gratuit"].value_counts(normalize=True) * 100
df_filtre = df_filtre.drop(columns=["gratuit"])

#### paiement_cb
Tout comme `paiement_autre`, cette variable présente une faible proportion de valeurs manquantes (environ 8 %) et semble contenir une information intéressante.
Nous faisons donc le choix de la conserver pour la suite de l’analyse.

In [ ]:
df_filtre["paiement_cb"].value_counts(normalize=True) * 100

#### nom_operateur

La part de valeurs manquantes reste faible, cela ne nuira pas à notre analyse, nous décidons de garder cette variable pour le moment.

## 6. Choix des Variables

#### Variables écartées

Commençons par réaliser quelques analyses exploratoires rapides afin d’identifier et d’écarter d’éventuelles variables non pertinentes pour la suite de l’étude.

In [ ]:
(df_filtre.select_dtypes(include="bool")
 .drop(columns=["paiement_autre", "paiement_cb"])
 .apply(lambda x: x.value_counts(normalize=True) * 100))

In [ ]:
df_filtre["condition_acces"].value_counts(normalize=True) * 100

Les variables `paiement_acte`, `reservation`, `prise_type_chademo`, `prise_type_autre` et `condition_acces` présentent un fort déséquilibre entre leurs modalités, ce qui leur confère un faible pouvoir discriminant. Leur apport informationnel étant limité, nous faisons le choix de les supprimer.

In [ ]:
print(df_filtre["created_at"].min())
print(df_filtre["created_at"].max())
print("")
print(df_filtre["created_at"].dt.year.value_counts().sort_index())

`created_at` est davantage administrative que structurelle. Nous décidons de la supprimer.

In [ ]:
print(df_filtre["horaires"].value_counts(dropna=False).head(10))
print("")
print("Nb modalités uniques :", df_filtre["horaires"].nunique())

Cette variable contient des informations sous forme de texte libre. La grande diversité des modalités rend son exploitation difficile dans un cadre d’analyse ou de modélisation.
Par conséquent, nous faisons le choix de l’écarter de l’étude.

In [ ]:
colonnes_supp = ["horaires", "created_at", "paiement_acte", "reservation", "prise_type_chademo", "prise_type_autre", "condition_acces"]
df_filtre = df_filtre.drop(columns=colonnes_supp)

**Récapitulatif des variables écartées**

| Variable         | Raison                      |
|---|---|
| `cable_t2_attache` | Proportion importante de valeurs manquantes et modalités très déséquilibrées |
| `horaires` | Variable en texte libre, trop hétérogène pour être exploitée simplement |
| `tarification` | Nombre élevé de valeurs manquantes et contenu en texte libre très diversifié |
| `prise_type_autre` | Variable très déséquilibrée, faible apport informationnel |
| `prise_type_chademo` | Variable très déséquilibrée, faible apport informationnel |
| `paiement_acte` | Variable très déséquilibrée, pouvoir discriminant limité |
| `condition_acces` | Modalités fortement déséquilibrées, faible variabilité |
| `gratuit` | Quasi-totalité des bornes payantes, information peu discriminante |
| `reservation` | Variable très déséquilibrée, faible variabilité |
| `created_at` | Variable à caractère administratif plutôt que structurel |
| `nom_amenageur` | Trop grand nombre de modalités distinctes |
| `nom_enseigne` | Trop grand nombre de modalités distinctes |

Les variables supprimées présentent principalement l’un ou plusieurs des problèmes suivants :

- trop de valeurs manquantes ;
- modalités fortement déséquilibrées ;
- trop grand nombre de catégories ;
- contenu textuel libre difficile à exploiter ;
- faible intérêt analytique dans le cadre de l’étude.

Leur retrait permet de simplifier le jeu de données et de conserver les variables les plus pertinentes pour la suite de l’analyse.

#### Variables retenues
Finalement, les variables retenues pour la modélisation sont les suivantes :

In [ ]:
vars_finales = ['code_geo_total',
               'nom_operateur',
               'implantation_station_clean',
               'nbre_pdc',
               'puissance_nominale',
               'prise_type_ef',
               'prise_type_2',
               'prise_type_combo_ccs',
               'paiement_cb',
               'paiement_autre']
df_agg = df_filtre[vars_finales]

## 7. Agrégation territoriale

À partir des variables retenues au niveau des bornes de recharge, nous procédons à une agrégation par zone géographique afin de construire une base de données à l’échelle communale.

L’objectif est d’obtenir une structure où **une ligne correspond à un code géographique (commune)**, ce qui permettra par la suite de fusionner cette base avec les autres tables de données et de modéliser le taux de véhicules électriques.

Cette étape transforme donc les données “bornes” en indicateurs territoriaux synthétiques.

Voici un récapitulatif des choix d’agrégation réalisés :

| Variables utilisées       | Variable finale                              | Construction        |
|--------------------------|-----------------------------------------------|---------------------|
| nbre_pdc                 | total_pdc                                     | somme               |
| puissance_nominale       | puissance_moyenne                             | moyenne             |
| puissance_nominale       | puissance_max                                 | max                 |
| puissance_nominale       | pct_charge_rapide                             | moyenne booléenne   |
| nom_operateur            | nb_operateurs                                 | nunique             |
| nom_operateur            | top_operateur                                 | mode                |
| prise_type_2             | pct_type_2                                    | moyenne booléenne   |
| prise_type_combo_ccs     | pct_combo_ccs                                 | moyenne booléenne   |
| prise_type_ef            | pct_type_ef                                   | moyenne booléenne   |
| paiement_cb              | pct_paiement_cb                               | moyenne booléenne   |
| paiement_autre           | pct_paiement_autre                            | moyenne booléenne   |
| implantation_station     | prive, public, rapide, voirie                 | dummies + moyenne   |

Le jeu de données obtenu constitue désormais une base territoriale synthétique, prête à être fusionnée avec les autres sources de données locales afin d’expliquer et modéliser le taux de véhicules électriques.

In [ ]:
from src.features import creer_features_irve
from src.cleaning import clean_irve_variables_finales
data_finales = clean_irve_variables_finales(df_agg)
data_finales = creer_features_irve(data_finales)

## 8. Autre

### Création de la variable 'borne rapide'
Nous avons fait des analyses univariées pour bien comprendre les puissances des bornes et  créez une variable boléenn 'borne rapide'.

In [ ]:
import matplotlib.pyplot as plt
# ----------------------------
# Statistiques générales
# ----------------------------
print("===== Statistiques =====")
print(df_filtre["puissance_nominale"].describe())
print()

# ----------------------------
# Valeurs les plus fréquentes
# ----------------------------
print("===== Top puissances =====")
print(df_filtre["puissance_nominale"].value_counts().head(10))
print()

# ----------------------------
# Zoom sur faibles puissances
# ----------------------------
plt.figure(figsize=(10,5))
df_filtre[df_filtre["puissance_nominale"] <= 100]["puissance_nominale"].hist(bins=40)
plt.title("Zoom sur puissances <= 100 kW")
plt.xlabel("Puissance (kW)")
plt.ylabel("Nombre")
plt.show()

# ----------------------------
# Boxplot
# ----------------------------
plt.figure(figsize=(10,3))
plt.boxplot(df_filtre["puissance_nominale"].dropna(), vert=False)
plt.title("Boxplot puissance nominale")
plt.show()

# ----------------------------
# Quantiles utiles
# ----------------------------
print("===== Quantiles =====")
print(df_filtre["puissance_nominale"].quantile([0.25,0.5,0.75,0.90,0.95,0.99]))
print()

# ----------------------------
# Parts selon seuils métier
# ----------------------------
for seuil in [7, 22, 43, 50, 100, 150]:
    pct = (df_filtre["puissance_nominale"] >= seuil).mean() * 100
    print(f"% bornes >= {seuil} kW : {pct:.2f}%")

La puissance nominale des bornes de recharge permet de caractériser leur vitesse de recharge. Dans un contexte IRVE, les valeurs observées doivent respecter des ordres de grandeur physiques et techniques connus.

#### Tranches de puissance attendues

On peut distinguer plusieurs catégories usuelles :

- **≤ 3 kW** : recharge très lente (cas rares, domestique ancien ou données erronées)
- **3 à 7 kW** : recharge lente (souvent domestique ou résidentielle)
- **7 à 22 kW** : recharge accélérée en courant alternatif (AC), très fréquente en voirie et parkings publics
- **22 à 50 kW** : recharge rapide (souvent en courant continu DC)
- **50 à 150 kW** : recharge rapide à très rapide (stations dédiées, axes routiers)
- **≥ 150 kW** : recharge ultra-rapide (autoroutes, hubs de recharge)

#### Valeurs aberrantes

Certaines valeurs ne sont pas cohérentes avec la réalité technique des infrastructures IRVE :

- **0 kW** : valeur invalide ou donnée manquante mal encodée
- **> 300–350 kW** : très rare et dépend de technologies spécifiques
- **50000 kW (ou valeurs extrêmes similaires)** : clairement aberrantes, probablement liées à une erreur de saisie ou d’unité (ex : W au lieu de kW)

Ces valeurs devraient être considérées comme des *outliers* et traitées (suppression ou correction) afin d’éviter tout biais dans les analyses.
Dans ce travail, ce traitement n’a pas été approfondi, mais il constitue une piste d’amélioration importante pour la robustesse des résultats.

### Définition d’une borne rapide

Dans le cadre de cette étude, nous considérons qu’une borne est dite **rapide** lorsque sa puissance est :

$$
\text{puissance nominale} \geq 43 \text{ kW}
$$
